## CAMP2Ex Dropsonde Plotting Code (Questions?  Email brodenkirch@wisc.edu)

#### Given a filepath (_dropsonde_folder_, see cell #2) that contains CAMP2Ex dropsonde ".nc" (NetCDF) files, this code will:

1. Loop through each dropsonde file to filter/QC the data and add all filtered/QCed dropsonde data to the given date's (_file_date_, see cell #2) created `final_dropsonde_YYYYMMDD.csv` file, saved to _day_folder_. (cell #3)

2. Make _height vs. time_ dropsonde moisture and wind availability plots for the given _file_date_, saved to _day_folder_. (cell #4)

3. Make theta, theta-e, and theta-v plots for each dropsonde profile, saved to _day_folder_. (cell #5)

4. Make skew-T plots (not quite publication quality) for each dropsonde profile using MetPy, saved to _day_folder_. (cell #6)

### Hope this helps!  Feel free to edit the code to fit your needs.  The variables you will definitely want to change right away are _file_date_ (cell #2), _day_folder_ (cell #2), and _dropsonde_folder_ (cell #2).  Once you change these, everything should run properly as is.  If it doesn't, let the author know (brodenkirch@wisc.edu). You likely will want to edit the dropsondes in the _sondes_with_nowind_or_nomoisture_ list as well (top of cell #3).

In [ ]:
import os
import sys
import xarray as xr
import pandas as pd
import numpy as np
from datetime import datetime

import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
#import matplotlib.colors as mplc

import sharppy
import sharppy.sharptab.profile as profile
#import sharppy.sharptab.interp as interp
import sharppy.sharptab.winds as winds
import sharppy.sharptab.utils as utils
import sharppy.sharptab.params as params
import sharppy.sharptab.thermo as thermo

import metpy.calc as mpcalc
import metpy.plots as mplots
from metpy.units import units

from PIL import Image
            

In [ ]:
file_date = '20190904'  #which date to create clean dropsonde data CSV for

day_folder = os.path.join(os.getcwd(), file_date)   
dropsonde_folder = os.path.join(day_folder, 'Dropsonde_files')
drop_final_name = os.path.join(day_folder, 'final_dropsonde_' + file_date + '.csv')

#Display the contents of a typical CAMP2Ex dropsonde NetCDF file
test = xr.open_dataset(os.path.join('/Users/ben/Desktop/CAMP2Ex/Coding/20190829/Dropsonde_files', 'CAMP2EX-dropsondes_P3B_20190829234615_R0.nc'))
test


In [ ]:
#loop through each dropsonde file to filter/QC the data and add to the given date's final_dropsonde CSV
sondes_with_nowind_or_nomoisture = ['20191005061720']

first_file = True
for a in sorted(os.listdir(dropsonde_folder)):  #sorted() goes through the files in alphabetical order
    
    if a[-3:] != '.nc':  #grab only the dropsonde .nc files from the directory
        continue
    else:
        ds = xr.open_dataset(os.path.join(dropsonde_folder, a))
        
    #convert the dataset to a Pandas dataframe
    df = ds.to_dataframe()
    
    #make the dropsonde time the time when the dropsonde was deployed, instead of the time of the first good data point
    drop_full_time = str(df['reference_time'].iloc[0])

    #Using the QC method for all sondes that would normally only be used for sondes with large gaps in wind/moisture data
        #because of the following from "CAMP2Ex-dropsondes_P3B_R1_README.pdf":
            #"74 of the 193 valid dropsondes with reported data failed to report valid wind data within 60 seconds of launch. These
            #dropsondes are described as having “Late Winds”. Although this issue was first noticed in the field, the root cause of the issue
            #was not uncovered until after the campaign. After the test flights, the GPS repeater located in the dropsonde tube became
            #disconnected from the primary GPS receiver in the forward cabin of the P-3B. This resulted in the dropsondes not receiving a GPS
            #signal while in the tube, which led to the failure to quickly acquire this signal after release. All in all, 72 of the 74
            #Late Winds dropsondes reported wind data before termination of data transmission, whereas 2 never reported winds while falling
            #to the ocean surface. Although these drops are missing some or all of their wind data, the thermodynamic data and initial launch
            #position are unaffected and can still be employed in data analysis."

    ##The QC method below is used only if no wind data or no moisture data (or at least very large gaps) exists throughout a dropsonde (see CAMP2Ex dropsonde README). 
    ###Otherwise, this QC method screws up metric calculations if u/v/temp/RH/dewpoint are not all present for a given line of data for a normal dropsonde
    #QC the data: alt. (hydrostatic or GPS) available and > 0, pressure > 0 (and not NaN), and (u/v wind not NaN) OR (temp/RH/dewpoint not NaN)
    df_use = df[((df['alt'] > 0) | (df['gpsalt'] > 0)) & (df['pres'] > 0) & 
                (((df['u_wind'].notnull()) & (df['v_wind'].notnull())) | ((df['tdry'].notnull()) & (df['dp'].notnull()) & (df['rh'] >= 0)))].copy()
        #^^^ using .copy() to prevent chain-indexing --> https://www.dataquest.io/blog/settingwithcopywarning/
    
    # if a[-20:-6] in sondes_with_nowind_or_nomoisture:
    # ##The QC method below is used only if no wind data or no moisture data (or at least very large gaps) exists throughout a dropsonde (see CAMP2Ex dropsonde README). 
    # ###Otherwise, this QC method screws up metric calculations if u/v/temp/RH/dewpoint are not all present for a given line of data for a normal dropsonde
    #     #QC the data: alt. (hydrostatic or GPS) available and > 0, pressure > 0 (and not NaN), and (u/v wind not NaN) OR (temp/RH/dewpoint not NaN)
    #     df_use = df[((df['alt'] > 0) | (df['gpsalt'] > 0)) & (df['pres'] > 0) & 
    #                 (((df['u_wind'].notnull()) & (df['v_wind'].notnull())) | ((df['tdry'].notnull()) & (df['dp'].notnull()) & (df['rh'] >= 0)))].copy()
    #         #^^^ using .copy() to prevent chain-indexing --> https://www.dataquest.io/blog/settingwithcopywarning/
    # else:
    #     #QC the data: alt. (hydrostatic or GPS) available and > 0, pressure > 0 (and not NaN), u/v/temp/RH/dewpoint not NaN
    #     df_use = df[((df['alt'] > 0) | (df['gpsalt'] > 0)) & (df['pres'] > 0) & (df['u_wind'].notnull()) & 
    #                 (df['v_wind'].notnull()) & (df['tdry'].notnull()) & (df['dp'].notnull()) & (df['rh'] >= 0)].copy()
    #         #^^^ using .copy() to prevent chain-indexing --> https://www.dataquest.io/blog/settingwithcopywarning/

    if len(df_use) != 0:
        #grab the time of the first good data line (the last index), to be used as the official dropsonde time
        #drop_full_time = str(df_use.index[-1][0])[:19]

        #create one single height column, prioritizing hydrostatic altitude over GPS altitude
        heights_use = []
        for i in range(len(df_use)):
            hydro_height = df_use['alt'].iloc[i]
            if hydro_height > 0:  #i.e., if the hydrostatic height value is not NaN, use hydrostatic height
                heights_use.append(hydro_height)
            else:
                heights_use.append(df_use['gpsalt'].iloc[i])  #if the hydrostatic height value is NaN, use GPS height
        df_use['Heights Use'] = heights_use  #needed for proper rounding of height Series....for some reason

        #convert the good, relevant dropsonde data to a dataframe and add to the final dropsonde CSV file           
        drop_clean_df = pd.DataFrame(columns = ['Time [UTC]', 'Height [m]', 'Pressure [mb]', 'U Comp of Wind [m/s]', 'V Comp of Wind [m/s]',
                                                'Wind Speed [m/s]', 'Wind Direction [deg]', 'Temperature [C]', 'Dew Point [C]',
                                                'Potential Temperature [K]', 'Relative Humidity [%]', 'Latitude [deg]', 'Longitude [deg]'])
        drop_clean_df['Time [UTC]'] = [drop_full_time] * len(df_use)  #a list of len(df_use) with the same drop_full_time value
        drop_clean_df['Height [m]'] = np.round(list(df_use['Heights Use'])[::-1], 2)
        drop_clean_df['Pressure [mb]'] = np.round(list(df_use['pres'])[::-1], 2)
        drop_clean_df['U Comp of Wind [m/s]'] = np.round(list(df_use['u_wind'])[::-1], 2)
        drop_clean_df['V Comp of Wind [m/s]'] = np.round(list(df_use['v_wind'])[::-1], 2)
        drop_clean_df['Wind Speed [m/s]'] = np.round(list(df_use['wspd'])[::-1], 2)
        drop_clean_df['Wind Direction [deg]'] = np.round(list(df_use['wdir'])[::-1], 2)
        drop_clean_df['Temperature [C]'] = np.round(list(df_use['tdry'])[::-1], 2)
        drop_clean_df['Dew Point [C]'] = np.round(list(df_use['dp'])[::-1], 2)
        drop_clean_df['Potential Temperature [K]'] = np.round(list(df_use['theta'])[::-1], 2)
        drop_clean_df['Relative Humidity [%]'] = np.round(list(df_use['rh'])[::-1], 2)
        drop_clean_df['Latitude [deg]'] = np.round(list(df_use['lat'])[::-1], 7)
        drop_clean_df['Longitude [deg]'] = np.round(list(df_use['lon'])[::-1], 7)

        if first_file:
            drop_clean_df.to_csv(drop_final_name, index = False)
            first_file = False
        else:
            df_all = pd.read_csv(drop_final_name)
            df_total = pd.concat([df_all, drop_clean_df], ignore_index = True)  #concatenates fields with same heading
            df_total.to_csv(drop_final_name, index = False)

    ds.close()
            

In [ ]:
#plot up the available, good dropsonde data for the given time range
df_all = pd.read_csv(drop_final_name)

df_winds = df_all[(df_all['U Comp of Wind [m/s]'].notnull()) & (df_all['V Comp of Wind [m/s]'].notnull())].copy()
df_moisture = df_all[(df_all['Temperature [C]'].notnull()) & (df_all['Dew Point [C]'].notnull()) & (df_all['Relative Humidity [%]'].notnull())].copy()

drop_fig, drop_axs = plt.subplots(nrows=1, ncols=2, figsize = (35,20))
drop_fig.subplots_adjust(wspace=0.2)

drop_moisture_x_ax = pd.to_datetime(df_moisture['Time [UTC]'])
drop_moisture_y_ax = df_moisture['Height [m]']
drop_axs[0].scatter(drop_moisture_x_ax, drop_moisture_y_ax, s=15, c='k')
drop_axs[0].set_ylim([0, 8500])
drop_axs[0].set_yticks(np.arange(0, 8501, 500))
drop_axs[0].tick_params(axis='x', rotation = 50)
drop_axs[0].tick_params(labelsize=18)
drop_axs[0].grid(True)
drop_axs[0].set_xlabel('Time [UTC]', fontsize=30)
drop_axs[0].set_ylabel('Height [m]', fontsize=30)
drop_axs[0].set_title('Dropsonde Moisture Availability', fontsize=40)
drop_axs[0].xaxis.set_major_formatter(mpl.dates.DateFormatter("%H:%M"))
#drop_axs[0].gcf().set_size_inches(10,13)

drop_winds_x_ax = pd.to_datetime(df_winds['Time [UTC]'])
drop_winds_y_ax = df_winds['Height [m]']
drop_axs[1].scatter(drop_winds_x_ax, drop_winds_y_ax, s=15, c='k')
drop_axs[1].set_ylim([0, 8500])
drop_axs[1].set_yticks(np.arange(0, 8501, 500))
drop_axs[1].tick_params(axis='x', rotation = 50)
drop_axs[1].tick_params(labelsize=18)
drop_axs[1].grid(True)
drop_axs[1].set_xlabel('Time [UTC]', fontsize=30)
drop_axs[1].set_ylabel('Height [m]', fontsize=30)
drop_axs[1].set_title('Dropsonde Wind Availability', fontsize=40)
drop_axs[1].xaxis.set_major_formatter(mpl.dates.DateFormatter("%H:%M"))
#drop_axs[1].gcf().set_size_inches(10,13)

# #plot up same figure but with wind barbs instead of dots
# drop_u = df_all['U Comp of Wind [m/s]']
# drop_v = df_all['V Comp of Wind [m/s]']
# drop_axs[1].barbs(drop_x_ax, drop_y_ax, drop_u, drop_v, fill_empty = True, pivot='middle', sizes=dict(emptybarb=0.075), barbcolor = 'b')
# #add "np.sqrt(drop_u**2 + drop_v**2)" to above line to color code barbs by speed
# drop_axs[1].tick_params(axis='x', rotation = 50)
# drop_axs[1].tick_params(labelsize=18)
# drop_axs[1].grid(True)
# drop_axs[1].set_xlabel('Time [UTC]', fontsize=30)
# drop_axs[1].set_ylabel('Height [m]', fontsize=30)
# drop_axs[1].set_title('Dropsonde 2-D Wind at Given Times and Heights', fontsize=40)
# drop_axs[1].xaxis.set_major_formatter(mpl.dates.DateFormatter("%H:%M"))
# #drop_axs[1].gcf().set_size_inches(20,25)

drop_name = os.path.join(day_folder, "Dropsonde_avail_moisture_and_winds_" + file_date + ".png")
plt.savefig(drop_name, bbox_inches = 'tight')  #bbox_inches = 'tight' will clip any additional white space around the image
plt.close()


In [ ]:
#make theta, theta-e, and virtual theta plots for each dropsonde to figure out which ones should be omitted
#exclude dropsonde profiles with large vertical data gaps and/or frequent, graphically visible anomalous spikes
    
drop_csv = pd.read_csv(drop_final_name)
drop_times = sorted(drop_csv['Time [UTC]'].unique())  #sorted() = goes through files in alphabetical order

line_types = ['b-', 'r-', 'k-', 'y-', 'b--', 'r--', 'k--', 'y--', 'c-', 'c--', 'm-', 'm--', 'b:', 'r:', 'k:', 'y:', 'c:', 'm:', 'b-.', 'r-.', 'k-.', 'y-.', 'c-.', 'm-.', 'g-', 'g--', 'g:', 'g-.',
              'b-', 'r-', 'k-', 'y-', 'b--', 'r--', 'k--', 'y--', 'c-', 'c--', 'm-', 'm--', 'b:', 'r:', 'k:', 'y:', 'c:', 'm:', 'b-.', 'r-.', 'k-.', 'y-.', 'c-.', 'm-.', 'g-', 'g--', 'g:', 'g-.']

#Potential Temperature 
fig = plt.figure(figsize=(15,15))   
    
line_index = 0
for time in drop_times:
    rel_data = drop_csv[drop_csv['Time [UTC]'] == time].copy()

    pres = rel_data['Pressure [mb]']
    hght = rel_data['Height [m]']
    tmpc = rel_data['Temperature [C]']
    dwpc = rel_data['Dew Point [C]']
    wspd = 1.94384449 * rel_data['Wind Speed [m/s]']  #converts m/s to knots (also in SHARPpy sharptab.utils script)
    wdir = rel_data['Wind Direction [deg]']

    plt.plot(rel_data['Potential Temperature [K]'], rel_data['Pressure [mb]'], line_types[line_index], label = time[11:])
    plt.xlabel("Potential Temperature [K]", fontsize = 25)
    plt.ylabel("Pressure [mb]", fontsize = 25)
    plt.ylim([1050,190])  #inverts y-axis (pressure)
    plt.yticks(np.arange(1000,190,-50))
    #plt.xlim([290, 380])
    #plt.xticks(np.arange(290,380.1,10))
    plt.tick_params(labelsize = 15)
    plt.legend(fontsize = 'xx-large')
    plt.grid(True)
    plt.title(file_date + ' Dropsonde Theta Profiles', fontsize = 30)
    plt.savefig(os.path.join(day_folder, 'theta_profiles.png'), bbox_inches = 'tight')  #bbox_inches = 'tight' will clip any additional white space around the image
    line_index = line_index + 1
plt.close()
  

#Equivalent Potential Temperature
fig = plt.figure(figsize=(15,15))

line_index = 0
for time in drop_times:
    rel_data = drop_csv[drop_csv['Time [UTC]'] == time].copy()
    rel_data2 = rel_data.iloc[::-1]  #reverses the dataframe (row-based) to go from surface to upper-level

    pres = rel_data2['Pressure [mb]']
    hght = rel_data2['Height [m]']
    tmpc = rel_data2['Temperature [C]']
    dwpc = rel_data2['Dew Point [C]']
    wspd = 1.94384449 * rel_data2['Wind Speed [m/s]']  #converts m/s to knots (also in SHARPpy sharptab.utils script)
    wdir = rel_data2['Wind Direction [deg]']

    try:
        prof = profile.create_profile(profile='default', pres=pres, hght=hght, tmpc=tmpc, dwpc=dwpc, wspd=wspd, wdir=wdir, missing=-9999, strictQC=True)
    except:
        prof = profile.create_profile(profile='default', pres=pres, hght=hght, tmpc=tmpc, dwpc=dwpc, wspd=wspd, wdir=wdir, missing=-9999, strictQC=False)
            
    plt.plot(prof.thetae.data, prof.pres.data, line_types[line_index], label = time[11:])
    plt.xlabel("Equivalent Potential Temperature [K]", fontsize = 25)
    plt.ylabel("Pressure [mb]", fontsize = 25)
    plt.ylim([1050,190])  #inverts y-axis (pressure)
    plt.yticks(np.arange(1000,190,-50))
    #plt.xlim([320, 380])
    #plt.xticks(np.arange(320,380.1,5))
    plt.tick_params(labelsize = 15)
    plt.legend(fontsize = 'xx-large')
    plt.grid(True)
    plt.title(file_date + ' Dropsonde Theta-E Profiles', fontsize = 30)
    plt.savefig(os.path.join(day_folder, 'thetaE_profiles.png'), bbox_inches = 'tight')  #bbox_inches = 'tight' will clip any additional white space around the image
    line_index = line_index + 1
plt.close()
    
    
#Virtual Potential Temperature    
fig = plt.figure(figsize=(15,15))
    
line_index = 0
for time in drop_times:
    rel_data = drop_csv[drop_csv['Time [UTC]'] == time].copy()
    rel_data2 = rel_data.iloc[::-1]  #reverses the dataframe (row-based) to go from surface to upper-level

    pres = rel_data2['Pressure [mb]']
    hght = rel_data2['Height [m]']
    tmpc = rel_data2['Temperature [C]']
    dwpc = rel_data2['Dew Point [C]']
    wspd = 1.94384449 * rel_data2['Wind Speed [m/s]']  #converts m/s to knots (also in SHARPpy sharptab.utils script)
    wdir = rel_data2['Wind Direction [deg]']

    try:
        prof = profile.create_profile(profile='default', pres=pres, hght=hght, tmpc=tmpc, dwpc=dwpc, wspd=wspd, wdir=wdir, missing=-9999, strictQC=True)
    except:
        prof = profile.create_profile(profile='default', pres=pres, hght=hght, tmpc=tmpc, dwpc=dwpc, wspd=wspd, wdir=wdir, missing=-9999, strictQC=False)

    thetav = thermo.theta(prof.pres.data, thermo.virtemp(prof.pres.data, prof.tmpc.data, prof.dwpc.data))   
    thetav = thermo.ctok(thetav)  #convert from Celsius to Kelvin
    plt.plot(thetav, prof.pres.data, line_types[line_index], label = time[11:])
    plt.xlabel("Virtual Potential Temperature [K]", fontsize = 25)
    plt.ylabel("Pressure [mb]", fontsize = 25)
    plt.ylim([1050,190])  #inverts y-axis (pressure)
    plt.yticks(np.arange(1000,190,-50))
    #plt.xlim([290, 380])
    #plt.xticks(np.arange(290,380.1,10))
    plt.tick_params(labelsize = 15)
    plt.legend(fontsize = 'xx-large')
    plt.grid(True)
    plt.title(file_date + ' Dropsonde Theta-V Profiles', fontsize = 30)
    plt.savefig(os.path.join(day_folder, 'thetaV_profiles.png'), bbox_inches = 'tight')  #bbox_inches = 'tight' will clip any additional white space around the image
    line_index = line_index + 1
plt.close()


In [ ]:
#plot dropsonde skew-T using MetPy(don't need to change anything in this cell)
def plot_skewTs(file_date, plot_hodograph = True):
    """ Make skew-T/hodograph figures for a given day's dropsonde profiles
    
    PARAMETERS
    ----------
    file_date : the day (YYYYMMDD, string format) for which you want to plot dropsonde profile skew-Ts
    plot_hodograph : determines whether or not to plot an inset hodograph (True/False)
    
    RETURNS
    ----------
    fig : matplotlib skew-T figures for each dropsonde in the given day's "final_dropsonde_YYYYMMDD.csv" file
    
    """
    
    day_folder = os.path.join(os.getcwd(), file_date)   
    dropsonde_folder = os.path.join(day_folder, 'Dropsonde_files')
    drop_final_name = os.path.join(day_folder, 'final_dropsonde_' + file_date + '.csv')
    
    drop_csv = pd.read_csv(drop_final_name)
    drop_times = drop_csv['Time [UTC]'].unique()  #sorted() = goes through files in alphabetical order
    
    #initialize some plot visualizations
    mpl.rcParams['font.family'] = 'arial'
    mpl.rcParams['font.size'] = 15
    mpl.rcParams['ytick.labelsize'] = 14
    mpl.rcParams['xtick.labelsize'] = 14

    for time in drop_times:
        #print (time)    #for debugging purposes
        
        df0 = drop_csv[drop_csv['Time [UTC]'] == time].copy()
        df = df0.iloc[::-1]  #reverses the dataframe (row-based) to go from surface to upper-level

        if time == '2019-09-04 03:49:03':   #surface moisture is missing, which causes skew-T to not be calculated/plotted
            #add appropriate units to the pressure, temperature, dewpoint, and wind data
            pres = df['Pressure [mb]'].values[1:] * units.hPa   #hPa = mb
            #filtered_pres = pres[::2][1:]      #only every other pressure value for quicker profile/skew-T creation
            temp = df['Temperature [C]'].values[1:] * units.degC
            #filtered_temp = temp[::2][1:]      ##only every other temp. value for quicker skew-T creation
            dwpt = df['Dew Point [C]'].values[1:] * units.degC
            #wnd_spd = df['Wind Speed [m/s]'].values[1:] * units('m/s')
            wnd_spd = (df['Wind Speed [m/s]'].values[1:] * units('m/s')).to(units.knots)  #convert wind speed to knots
            wnd_dir = df['Wind Direction [deg]'].values[1:] * units.deg
            #hght = df['Height [m]'].values[1:] * units.meter
        else:
            #add appropriate units to the pressure, temperature, dewpoint, and wind data
            pres = df['Pressure [mb]'].values * units.hPa   #hPa = mb
            #filtered_pres = pres[::2]      #only every other pressure value for quicker profile/skew-T creation
            temp = df['Temperature [C]'].values * units.degC
            #filtered_temp = temp[::2]      ##only every other temp. value for quicker skew-T creation
            dwpt = df['Dew Point [C]'].values * units.degC
            #wnd_spd = df['Wind Speed [m/s]'].values * units('m/s')
            wnd_spd = (df['Wind Speed [m/s]'].values * units('m/s')).to(units.knots)  #convert wind speed to knots
            wnd_dir = df['Wind Direction [deg]'].values * units.deg
            #hght = df['Height [m]'].values * units.meter

        #calculate the parcel path/profile at the near-surface for the given environment
        
        #profile = mpcalc.parcel_profile(pres, temp[0], dwpt[0])  #returns temps in Kelvin
        try:
            profile = mpcalc.parcel_profile(pres, temp[0], dwpt[0])  #returns temps in Kelvin
        except:
            print (f'Could not plot {time} skew-T, likely because this dropsonde contains no moisture data or pressure increases between at least two points in the sounding. Using scipy.signal.medfilt may fix the latter.')
            continue
            
        profile = profile.to('degC')

        #calculate the LCL and wind components
        lcl = mpcalc.lcl(pres[0], temp[0], dwpt[0])
        wind_comps = mpcalc.wind_components(wnd_spd, wnd_dir)  #returns u,v values in whatever unit wnd_spd is in
        u = wind_comps[0]
        v = wind_comps[1]

        #initialize the figure
        fig = plt.figure(figsize = (12,12))       

        #Initialize the skew-T figure/subplot
        skew = mplots.SkewT(fig)

        #Plot the data for the skew-T
        skew.plot(pres, temp, 'darkorange', linewidth = 2)
        skew.plot(pres, dwpt, 'cornflowerblue', linewidth = 2)
        #skew.plot(lcl[0], lcl[1], 'yellow', marker = '*', markeredgecolor = 'k', markersize = 14)  #plot the LCL as a yellow star
        #skew.plot(filtered_pres, profile, 'k', linewidth = 2)
        skew.plot(pres, profile, 'k', linewidth = 2)
        skew.plot_barbs(pres[::60], u[::60], v[::60])
        #skew.shade_cape(filtered_pres, filtered_temp, profile)
        #skew.shade_cin(filtered_pres, filtered_temp, profile, dwpt[::2])
        skew.shade_cape(pres, temp, profile, alpha = 0.2)
        skew.shade_cin(pres, temp, profile, alpha = 0.2)
        skew.plot_dry_adiabats(t0 = np.arange(-90, 321, 10) * units.degC, alpha = 0.3)   #range is large to cover whole plot for all possible profiles
        skew.plot_moist_adiabats(t0 = np.arange(-90, 81, 10) * units.degC, alpha = 0.3)  #range is large to cover whole plot for all possible profiles
        skew.ax.set_xlim(-50,40)
        skew.ax.set_ylim(1000,200)
        skew.ax.set_title(f'{time} Dropsonde Skew-T Diagram and Hodograph') #title created based on dropsonde time
        skew.ax.set_xlabel('T [$\\degree$C]')
        skew.ax.set_ylabel('Pressure [hPa]')
        
        #if just one sounding text file is inputted, then plot a hodograph in the upper right-hand corner of the figure
        if plot_hodograph:
            axh = inset_axes(skew.ax, '35%', '35%', loc = 'upper right')
            h = mplots.Hodograph(axh, component_range = 80.)
            h.add_grid(increment = 20)
            
            try:
                h.plot_colormapped(u, v, wnd_spd);  # Plot a line colored by wind speed
            except:
                fig.text(0.755, 0.643, 'No Wind Data', horizontalalignment='center', 
                         verticalalignment='center', fontsize = 20)
        
        save_name = os.path.join(day_folder, 'skewt_' + time[11:13] + time[14:16] + time[17:19] + '.png')
        #plt.show()
        plt.savefig(save_name, bbox_inches = 'tight')  #bbox_inches = 'tight' will clip any additional white space around the image
        plt.close()
        
        #decrease file size of the image by 4x without noticeable image effects (if using Matplotlib)!
        #(good to use if you're producing a lot of images, see https://www.youtube.com/watch?v=fzhAseXp5B4)
        im = Image.open(save_name)
        try:
            im2 = im.convert('P', palette = Image.Palette.ADAPTIVE)
        except:
            im2 = im.convert('P')  #use this for older version of PIL/Pillow if the above line doesn't work, though this line will have isolated, extremely minor image effects due to only using 256 colors instead of the 3-part RGB scale
        im2.save(save_name)
        im.close()
        im2.close()

plot_skewTs(file_date)


In [ ]:
print ('Done!')